# OTEL Spans — Silver Pipeline

Spark Declarative Pipeline that creates silver tables from raw OTEL span data.

| Table | Kind | Description |
| --- | --- | --- |
| `spans_flattened` | Streaming | All spans with timestamps, duration, K8s hierarchy, and service identity |
| `spans_service_calls` | Streaming | HTTP & gRPC server spans with route, status code, and latency |
| `spans_errors` | Streaming | Error spans for monitoring and alerting |
| `spans_db_operations` | Streaming | Database spans (PostgreSQL, Redis) with query text |
| `spans_messaging` | Streaming | Kafka producer/consumer spans with partition, offset, and queue lag |
| `spans_service_dependency_map` | Materialized View | Caller→callee edges with call counts and avg duration |

In [ ]:
import dlt
from pyspark.sql import functions as F

source_catalog = spark.conf.get("source_catalog")
source_schema = spark.conf.get("source_schema")
table_prefix = spark.conf.get("table_prefix")
source_table = f"{source_catalog}.{source_schema}.{table_prefix}_otel_spans"

In [ ]:
@dlt.table(
    comment="All OTel spans with flattened attributes, duration, and K8s resource hierarchy",
)
def spans_flattened():
    return (
        spark.readStream.table(source_table)
        .filter(F.col("start_time_unix_nano").isNotNull())
        .selectExpr(
            "trace_id",
            "span_id",
            "parent_span_id",
            "name AS span_name",
            "kind",
            "TIMESTAMP_MICROS(CAST(start_time_unix_nano / 1000 AS BIGINT)) AS start_timestamp",
            "TIMESTAMP_MICROS(CAST(end_time_unix_nano / 1000 AS BIGINT)) AS end_timestamp",
            "ROUND((end_time_unix_nano - start_time_unix_nano) / 1e6, 2) AS duration_ms",
            "status.code AS status_code",
            "status.message AS status_message",
            # K8s resource hierarchy
            "resource.attributes['k8s.node.name'] AS node",
            "resource.attributes['k8s.namespace.name'] AS namespace",
            "resource.attributes['k8s.deployment.name'] AS deployment",
            "resource.attributes['k8s.pod.name'] AS pod",
            "resource.attributes['k8s.pod.uid'] AS pod_uid",
            "resource.attributes['k8s.container.name'] AS container",
            "resource.attributes['k8s.cluster.uid'] AS cluster_uid",
            # Service identity
            "resource.attributes['service.name'] AS service",
            "resource.attributes['service.namespace'] AS service_namespace",
            "resource.attributes['service.instance.id'] AS service_instance_id",
            "resource.attributes['service.version'] AS service_version",
            # Instrumentation
            "instrumentation_scope.name AS instrumentation_name",
            "instrumentation_scope.version AS instrumentation_version",
            # Common span attributes
            "attributes['http.request.method'] AS http_method",
            "COALESCE(attributes['http.route'], attributes['url.path'], attributes['http.target']) AS http_route",
            "CAST(COALESCE(attributes['http.response.status_code'], attributes['http.status_code']) AS INT) AS http_status_code",
            "attributes['rpc.system'] AS rpc_system",
            "attributes['rpc.service'] AS rpc_service",
            "attributes['rpc.method'] AS rpc_method",
            "CAST(attributes['rpc.grpc.status_code'] AS INT) AS grpc_status_code",
            "attributes['db.system'] AS db_system",
            "attributes['messaging.system'] AS messaging_system",
            "attributes['error.type'] AS error_type",
            "attributes",
        )
    )

In [ ]:
@dlt.table(
    comment="HTTP and gRPC server spans — one row per inbound request with route, status, and latency",
)
def spans_service_calls():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("kind") == "SPAN_KIND_SERVER")
        .select(
            "trace_id",
            "span_id",
            "parent_span_id",
            "start_timestamp",
            "end_timestamp",
            "duration_ms",
            "status_code",
            "status_message",
            "service",
            "service_version",
            "node",
            "namespace",
            "deployment",
            "pod",
            # HTTP fields
            "span_name",
            "http_method",
            "http_route",
            "http_status_code",
            # gRPC fields
            "rpc_system",
            "rpc_service",
            "rpc_method",
            "grpc_status_code",
            # Protocol classification
            F.when(F.col("rpc_system").isNotNull(), F.lit("grpc"))
             .when(F.col("http_method").isNotNull(), F.lit("http"))
             .otherwise(F.lit("other"))
             .alias("protocol"),
            # Is this an error?
            F.when(
                (F.col("status_code") == "STATUS_CODE_ERROR")
                | (F.col("http_status_code") >= 500)
                | (F.col("grpc_status_code").isin(2, 13, 14)),  # UNKNOWN, INTERNAL, UNAVAILABLE
                F.lit(True),
            ).otherwise(F.lit(False)).alias("is_error"),
        )
    )

In [ ]:
@dlt.table(
    comment="Error spans for monitoring and alerting — any span with STATUS_CODE_ERROR",
)
def spans_errors():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("status_code") == "STATUS_CODE_ERROR")
        .select(
            "trace_id",
            "span_id",
            "parent_span_id",
            "span_name",
            "kind",
            "start_timestamp",
            "duration_ms",
            "status_message",
            "error_type",
            "service",
            "service_version",
            "node",
            "namespace",
            "deployment",
            "pod",
            "http_method",
            "http_route",
            "http_status_code",
            "rpc_service",
            "rpc_method",
            "grpc_status_code",
        )
    )

In [ ]:
@dlt.table(
    comment="Database operation spans (PostgreSQL, Redis) with query text and latency",
)
def spans_db_operations():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("db_system").isNotNull())
        .selectExpr(
            "trace_id",
            "span_id",
            "parent_span_id",
            "span_name",
            "start_timestamp",
            "duration_ms",
            "status_code",
            "status_message",
            "service",
            "node",
            "namespace",
            "pod",
            "db_system",
            "COALESCE(attributes['db.namespace'], attributes['db.name']) AS db_name",
            "COALESCE(attributes['db.query.text'], attributes['db.statement']) AS db_query",
            "attributes['db.user'] AS db_user",
            "CAST(attributes['db.redis.database_index'] AS INT) AS redis_db_index",
            "attributes['server.address'] AS server_address",
            "CAST(attributes['server.port'] AS INT) AS server_port",
        )
    )

In [ ]:
@dlt.table(
    comment="Kafka producer and consumer spans with partition, offset, and queue lag",
)
def spans_messaging():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("messaging_system").isNotNull())
        .selectExpr(
            "trace_id",
            "span_id",
            "parent_span_id",
            "span_name",
            "kind",
            "start_timestamp",
            "duration_ms",
            "status_code",
            "service",
            "node",
            "namespace",
            "pod",
            "messaging_system",
            "attributes['messaging.operation'] AS messaging_operation",
            "attributes['messaging.destination.name'] AS destination",
            "CAST(COALESCE(attributes['messaging.destination.partition.id'], attributes['messaging.kafka.destination.partition']) AS INT) AS partition_id",
            "CAST(attributes['messaging.kafka.message.offset'] AS BIGINT) AS message_offset",
            "attributes['messaging.kafka.consumer.group'] AS consumer_group",
            "attributes['messaging.client_id'] AS client_id",
            "CAST(attributes['messaging.message.body.size'] AS INT) AS message_body_size",
            "CAST(attributes['messaging.batch.message_count'] AS INT) AS batch_message_count",
            "CAST(attributes['kafka.record.queue_time_ms'] AS DOUBLE) AS queue_time_ms",
        )
    )

In [ ]:
@dlt.table(
    comment="Service dependency edges — caller to callee with call counts and avg duration",
)
def spans_service_dependency_map():
    # CLIENT spans represent outbound calls: the resource's service is the caller,
    # and the peer service (or rpc_service) is the callee.
    client_spans = (
        spark.read.table(source_table)
        .filter(
            (F.col("kind") == "SPAN_KIND_CLIENT")
            & F.col("start_time_unix_nano").isNotNull()
        )
        .selectExpr(
            "resource.attributes['service.name'] AS caller_service",
            "COALESCE("
            "  attributes['peer.service'],"
            "  attributes['rpc.service'],"
            "  attributes['db.system'],"
            "  attributes['messaging.destination.name'],"
            "  attributes['server.address'],"
            "  attributes['net.peer.name'],"
            "  name"
            ") AS callee_service",
            "CASE"
            "  WHEN attributes['rpc.system'] IS NOT NULL THEN 'grpc'"
            "  WHEN attributes['db.system'] IS NOT NULL THEN 'database'"
            "  WHEN attributes['messaging.system'] IS NOT NULL THEN 'messaging'"
            "  WHEN COALESCE(attributes['http.request.method'], attributes['http.method']) IS NOT NULL THEN 'http'"
            "  ELSE 'other'"
            " END AS protocol",
            "ROUND((end_time_unix_nano - start_time_unix_nano) / 1e6, 2) AS duration_ms",
            "CASE WHEN status.code = 'STATUS_CODE_ERROR' THEN 1 ELSE 0 END AS is_error",
        )
        .filter(F.col("caller_service").isNotNull() & F.col("callee_service").isNotNull())
    )

    return (
        client_spans
        .groupBy("caller_service", "callee_service", "protocol")
        .agg(
            F.count("*").alias("call_count"),
            F.round(F.avg("duration_ms"), 2).alias("avg_duration_ms"),
            F.round(F.expr("percentile(duration_ms, 0.5)"), 2).alias("p50_duration_ms"),
            F.round(F.expr("percentile(duration_ms, 0.95)"), 2).alias("p95_duration_ms"),
            F.round(F.expr("percentile(duration_ms, 0.99)"), 2).alias("p99_duration_ms"),
            F.sum("is_error").alias("error_count"),
            F.round(F.avg("is_error") * 100, 2).alias("error_rate_pct"),
        )
    )